<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Tag documents as you upload them. Ask a large library by its tags.**

Tags carry what a file does not say about itself. An SEC filing named by its accession number, such as `0001045810-24-000264.pdf`, does not say which company filed it, which form it is, or which fiscal quarter it covers, and fiscal calendars differ: NVIDIA's second quarter of fiscal 2025 ended July 28, 2024, while AMD's ended June 28, 2025. This notebook demonstrates how to declare a folder's metadata fields on PageIndex Cloud, tag documents as you upload them, and have the chat agent filter and count documents by those tags.

The notebook has two parts:

- **Part 1 · Tag documents on upload** — With your own PageIndex API key, declare a folder's fields and upload two 10-Q filings tagged with `company`, `form_type`, `fiscal_year`, and `fiscal_quarter`.
- **Part 2 · Ask a large library by tag** — With a read-only demo key, ask a library of 120 filings tagged with the same fields, and see the agent filter and count them by tag.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) and OpenAI API key ready. Part 1 creates a folder in your PageIndex Cloud library and uploads two filings into it.

# Part 1 · Tag documents on upload

This part uses **your own PageIndex API key**. It creates a folder in your library and uploads two filings into it.

## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the filings.

In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your PageIndex API key. The client stores your documents and their metadata on PageIndex Cloud.

In [1]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"

client = PageIndexClient(index="cloud")

## 02 · Declare the fields

Create a top-level folder and declare its fields. Each field has a type, `string`, `number`, or `boolean`, and a description that tells the agent what it holds. Subfolders share the fields. A folder's fields are set once; to change them, call `delete_metadata_schema()` and declare them again. A folder left by an earlier run is reused. `get_metadata_schema()` shows the fields the folder has.

In [2]:
from pageindex import PageIndexAPIError

FOLDER = "Metadata Demo Filings"
FIELDS = {
    "company": {"type": "string", "description": "Filing company: NVIDIA, AMD, Intel, Broadcom, Qualcomm, or Micron"},
    "form_type": {"type": "string", "description": "SEC form: 10-K (annual report) or 10-Q (quarterly report)"},
    "fiscal_year": {"type": "number", "description": "Fiscal year the company reports the filing under; the companies' fiscal years end in different months (NVIDIA's FY2025 ended Jan 26, 2025)"},
    "fiscal_quarter": {"type": "number", "description": "Fiscal quarter 1-3 of a 10-Q; a 10-K has none"},
}

try:
    folder_id = client.get_folder_id(FOLDER)  # created on an earlier run
except PageIndexAPIError:
    folder_id = client.create_folder(FOLDER)["folder"]["id"]
    client.create_metadata_schema(folder_id, FIELDS)
client.get_metadata_schema(folder_id)["schema"]

{'fields': {'company': {'type': 'string',
   'description': 'Filing company: NVIDIA, AMD, Intel, Broadcom, Qualcomm, or Micron'},
  'form_type': {'type': 'string',
   'description': 'SEC form: 10-K (annual report) or 10-Q (quarterly report)'},
  'fiscal_year': {'type': 'number',
   'description': "Fiscal year the company reports the filing under; the companies' fiscal years end in different months (NVIDIA's FY2025 ended Jan 26, 2025)"},
  'fiscal_quarter': {'type': 'number',
   'description': 'Fiscal quarter 1-3 of a 10-Q; a 10-K has none'}}}

## 03 · Tag documents on upload

Download two 10-Q filings from the PageIndex repository and submit each one into the folder with its tags. Neither file name says which company or quarter it covers; the tags do. `wait=True` returns once the document is ready. A filing already in the folder from an earlier run is reused rather than uploaded again.

In [3]:
import requests

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "0001045810-24-000264.pdf": {"company": "NVIDIA", "form_type": "10-Q", "fiscal_year": 2025, "fiscal_quarter": 2},
    "0000002488-25-000108.pdf": {"company": "AMD", "form_type": "10-Q", "fiscal_year": 2025, "fiscal_quarter": 2},
}

existing = {doc["name"]: doc["id"] for doc in client.list_documents(folder_id=folder_id)["documents"]}

doc_ids = {}
for file_name, metadata in DOCS.items():
    if file_name in existing:  # uploaded by an earlier run
        doc_ids[file_name] = existing[file_name]
        print(f"{file_name}: {doc_ids[file_name]} (already uploaded)")
        continue

    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(
        file_name, wait=True, folder_id=folder_id, metadata=metadata,
    )["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

0001045810-24-000264.pdf: pi-cmvp1ux200007h3nqlijt12tz
0000002488-25-000108.pdf: pi-cmvp1wppf003h0ap67h6d6dcc


The tags come back under `metadata` from `get_document()`, and on every document from `list_documents()`.


In [4]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

0001045810-24-000264.pdf: {'company': 'NVIDIA', 'form_type': '10-Q', 'fiscal_year': 2025, 'fiscal_quarter': 2}
0000002488-25-000108.pdf: {'company': 'AMD', 'form_type': '10-Q', 'fiscal_year': 2025, 'fiscal_quarter': 2}


Metadata is a flat dict of `str`, `int`, `float`, or `bool` values (numbers must be finite, so no `NaN` or `Infinity`), with keys that start with a letter and contain only letters, digits, and underscores. Its serialized size is at most 8 KB. A document may use only its folder's declared fields, each with its declared type; a folder without declared fields accepts no metadata. PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See the [metadata documentation](https://docs.pageindex.ai/sdk/documents#metadata) for details.


# Part 2 · Ask a large library by tag

A library of 120 filings is ready on PageIndex Cloud, tagged with the same four fields: the 10-K and 10-Q filings of NVIDIA, AMD, Intel, Broadcom, Qualcomm, and Micron, five fiscal years each, about 11,500 pages. Its files are named by accession number, like the two above. This part queries it with **a read-only demo key**, so nothing is uploaded.

## 04 · Connect to the library

Set your OpenAI API key and choose your chat `MODEL`. The `demo` client reads the library with the read-only demo key, and your chosen model answers with PageIndex Cloud's retrieval tools.

In [5]:
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

demo = PageIndexClient(
    index={"api_key": "pageindex_demo_readonly_key"},
    chat=MODEL,
)
LIBRARY_ID = "cmvoxrjpy002l0dnql7m69wh9"  # semiconductor-sec-filings

## 05 · Ask by tag

Ask for the third-quarter 10-Qs from fiscal 2024 onward. The agent reads the library's fields with `get_metadata_schema`, then filters by `form_type`, `fiscal_quarter`, and a `fiscal_year` range with `search_documents(mode="metadata")`. The output streams the answer alongside retrieval tool calls.

In [6]:
messages = [{"role": "user", "content": "List the third-quarter 10-Qs from fiscal 2024 onward, with each company and fiscal year."}]

response = None
for event in demo.chat(messages, folder_id=LIBRARY_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] browse_documents({"folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"sort":"relevance","query":"third quarter fiscal 2024 10-Q","limit":50})

[Tool call] browse_documents({"folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"sort":"relevance","query":"third quarter fiscal 2024 10-Q","offset":50,"limit":50})

[Tool call] get_metadata_schema({"folder_id":"cmvoxrjpy002l0dnql7m69wh9"})

[Tool call] search_documents({"mode":"metadata","folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"limit":50,"metadata":{"metadata_filter":{"form_type":"10-Q","fiscal_quarter":3,"fiscal_year":{"$gte":2024}},"sort":{"fiscal_year":1,"company":1}}})
Third-quarter Form 10-Q filings from fiscal 2024 onward:

| Fiscal year | Company | Filing |
|---|---|---|
| 2024 | AMD | `0000002488-24-000163.pdf` |
| 2024 | Broadcom | `0001730168-24-000099.pdf` |
| 2024 | Intel | `0000050863-24-000149.pdf` |
| 2024 | Micron | `0000723125-24-000019.pdf` |
| 2024 | NVIDIA | `0001045810-23-000227.pd

Tags also count. Ask how many 10-Ks and 10-Qs each company has: the agent groups the filings by `company` and `form_type` with `search_documents(mode="metadata")` and counts each group, without opening a filing.

In [7]:
messages = [{"role": "user", "content": "How many 10-K and 10-Q filings does each company have?"}]

response = None
for event in demo.chat(messages, folder_id=LIBRARY_ID, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] browse_documents({"folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"sort":"time","limit":50})

[Tool call] get_metadata_schema({"folder_id":"cmvoxrjpy002l0dnql7m69wh9"})

[Tool call] search_documents({"mode":"metadata","folder_id":"cmvoxrjpy002l0dnql7m69wh9","recursive":true,"limit":50,"metadata":{"group_by":["company","form_type"],"aggregates":{"filings":{"$count":{}}},"sort":{"company":1,"form_type":1}}})
| Company | 10-K filings | 10-Q filings | Total |
|---|---:|---:|---:|
| AMD | 5 | 15 | 20 |
| Broadcom | 5 | 15 | 20 |
| Intel | 5 | 15 | 20 |
| Micron | 5 | 15 | 20 |
| NVIDIA | 5 | 15 | 20 |
| Qualcomm | 5 | 15 | 20 |

Each company has **20 filings**: **5 annual 10-Ks** and **15 quarterly 10-Qs**.

## 06 · Review usage & estimated cost (optional)

Inspect the token usage for the last answer and estimate its model cost using the rate table below. Declaring fields, tagging documents, and reading the tags back do not require model calls. This estimate does not include PageIndex Cloud charges.


In [8]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 52206,
  "input_tokens_details": {
    "cached_tokens": 36062,
    "cache_write_tokens": 16132
  },
  "output_tokens": 403,
  "output_tokens_details": {
    "reasoning_tokens": 108
  },
  "total_tokens": 52609,
  "request_usage_entries": [
    {
      "input_tokens": 5462,
      "input_tokens_details": {
        "cached_tokens": 0,
        "cache_write_tokens": 5459
      },
      "output_tokens": 124,
      "output_tokens_details": {
        "reasoning_tokens": 76
      },
      "total_tokens": 5586
    },
    {
      "input_tokens": 15050,
      "input_tokens_details": {
        "cached_tokens": 5459,
        "cache_write_tokens": 9588
      },
      "output_tokens": 68,
      "output_tokens_details": {
        "reasoning_tokens": 32
      },
      "total_tokens": 15118
    },
    {
      "input_tokens": 15559,
      "input_tokens_details": {
        "cached_tokens": 15047,
        "cache_write_tokens": 509
      },
      "output_tokens": 80,
      "outpu

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
